# ResNet-20 Stage-wise Structured Pruning + Knowledge Distillation

We prune a CIFAR-10 ResNet-20 channel-wise (structured pruning) to reduce compute,
then measure how much accuracy fine-tuning (FT) and knowledge distillation (KD) can recover.

**Questions**

1. Which of ResNet-20's three stages (A, B, C) is most sensitive to pruning? (Part 1)
2. With a different pruning ratio per stage, can we get a model that is smaller than uniform 20% pruning
   while keeping similar accuracy? (Part 2)

**Environment**

- **Colab**: set the runtime type to **T4 GPU**. When cell 1 asks for Google Drive access, allow it;
  `data/`, `results/` and `figures/` are saved under `MyDrive/resnet20-stage-pruning-kd/`.
- **Local**: run `pip install -r requirements.txt` at the repo root, then run this notebook from `notebooks/`.

**Data & weights**: the CIFAR-10 subsets (10,000 train / 2,000 eval images) and the pretrained ResNet-20 weights
are downloaded at runtime from [imguru-mooc/AI_CAMPUS](https://github.com/imguru-mooc/AI_CAMPUS) (not included in this repo).

---

# Part 1 — Stage sensitivity: pruning shortcut paths only

Because of the shortcut (residual) connections, the output channels of all blocks in a stage are tied together.
We call each tied group a **shortcut path**; there is one per stage, three in total.

| Path | Scope | Channels |
|---|---|---|
| A | stem · layer1 | 16 |
| B | layer2 | 32 |
| C | layer3 (→ fc input) | 64 |

Each position of the tuple `(A, B, C)` is one path: `1` = prune 50%, `0` = keep.
Part 1 prunes **without any fine-tuning** to compare which path is more sensitive.

| Cell | Content | Time |
|---|---|---|
| 1 | Install & imports | 1–2 min (first run) |
| 2 | Load eval data & ResNet-20 | tens of seconds |
| 3 | `prune_paths` (prune by path tuple) | instant |
| 4 | Exp 1 (cumulative) · Exp 2 (single) | ~1 min |
| 5 | Plot (optional) | instant |
| 6 | Ratio sweep per path (optional) | ~1 min |

In [ ]:
# ▶ Cell 1 — Install & imports
import os, sys, copy, time, urllib.request, importlib.util, subprocess, warnings
# Install only missing packages (locally, installing via requirements.txt is recommended)
for mod, pkg in [("torch_pruning", "torch-pruning==1.6.0"), ("cv2", "opencv-python")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torch_pruning as tp
from IPython.display import display

warnings.filterwarnings("ignore")
torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
# Folder layout
#   Colab → save to Google Drive (persists after the runtime ends)
#   Local → if run inside notebooks/, the parent folder is the repo root
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/resnet20-stage-pruning-kd"
else:
    ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
DATA_DIR, RESULTS_DIR, FIG_DIR = (os.path.join(ROOT, d) for d in ("data", "results", "figures"))
for d in (DATA_DIR, RESULTS_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
print("Repo root:", ROOT)
print("PyTorch", torch.__version__, "| torch-pruning", getattr(tp, "__version__", "?"), "| device", DEV)

In [ ]:
# ▶ Cell 2 — CIFAR-10 eval set (2,000 images) · pretrained ResNet-20
BASE = "https://imguru-mooc.github.io/AI_CAMPUS/QP-day5"
RAW = "https://raw.githubusercontent.com/imguru-mooc/AI_CAMPUS/main/QP-day5"

def fetch(name):
    # Download data/<name> from the source repo if it does not exist
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        for url in [f"{BASE}/data/{name}", f"{RAW}/data/{name}"]:
            try:
                urllib.request.urlretrieve(url, path); break
            except Exception as e:
                if os.path.exists(path): os.remove(path)
                print("Download failed:", url, "→", e)
        else:
            raise FileNotFoundError(name)
    return path

MEAN = np.array((0.4914, 0.4822, 0.4465), np.float32)
STD = np.array((0.2470, 0.2435, 0.2616), np.float32)
def preprocess(images):
    x = images.astype(np.float32) / 255.0
    return ((x - MEAN) / STD).transpose(0, 3, 1, 2).copy()

ev = np.load(fetch("cifar_eval.npz"))
X_ev, y_ev = preprocess(ev["images"]), ev["labels"]

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, stride=1, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm2d(cout))
                         if stride != 1 or cin != cout else nn.Identity())
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet(nn.Module):
    def __init__(self, n=3, n_cls=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(*[BasicBlock(16, 16) for _ in range(n)])
        self.layer2 = nn.Sequential(BasicBlock(16, 32, 2), *[BasicBlock(32, 32) for _ in range(n - 1)])
        self.layer3 = nn.Sequential(BasicBlock(32, 64, 2), *[BasicBlock(64, 64) for _ in range(n - 1)])
        self.pool = nn.AdaptiveAvgPool2d(1); self.fc = nn.Linear(64, n_cls)
    def forward(self, x):
        x = self.layer3(self.layer2(self.layer1(self.stem(x))))
        return self.fc(self.pool(x).flatten(1))

def acc_torch(model, X=X_ev, y=y_ev, bs=500):
    model.eval(); dev = next(model.parameters()).device; preds = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            preds.append(model(torch.from_numpy(X[i:i + bs]).to(dev)).argmax(1).cpu().numpy())
    return float((np.concatenate(preds) == y).mean())

rn20 = ResNet(); rn20.load_state_dict(torch.load(fetch("resnet20_cifar10.pt"), map_location="cpu")); rn20.eval()
EX_IN = torch.randn(1, 3, 32, 32)       # example input for torch-pruning to trace the graph (only the shape matters)
FP32_ACC = acc_torch(rn20)
print(f"Eval set {X_ev.shape} | ResNet-20 FP32 accuracy {FP32_ACC:.4f}")

In [ ]:
# ▶ Cell 3 — Prune shortcut paths with a path tuple
PATH_ROOTS = [lambda m: m.stem[0],             # path A: stem output
              lambda m: m.layer2[0].conv2,     # path B: output of layer2's first block
              lambda m: m.layer3[0].conv2]     # path C: output of layer3's first block

def prune_paths(model, paths, ratio=0.5):
    m = copy.deepcopy(model).cpu().eval()                                   # ① prune a copy, keep the original
    DG = tp.DependencyGraph().build_dependency(m, example_inputs=EX_IN)    # ② dependency graph between layers
    imp = tp.importance.MagnitudeImportance(p=1)                           # ③ importance = L1-norm
    for on, get_root in zip(paths, PATH_ROOTS):                            # ④ one path at a time
        if not on:
            continue                                                       #    skip if 0
        root = get_root(m)                                                 # ⑤ representative conv of the path
        g_all = DG.get_pruning_group(root, tp.prune_conv_out_channels,
                                     idxs=list(range(root.out_channels)))  # ⑥ all layers tied to this path
        scores = imp(g_all)                                                # ⑦ importance per channel
        n_cut = int(round(root.out_channels * ratio))                      # ⑧ number of channels to prune
        idxs = scores.argsort()[:n_cut].tolist()                           # ⑨ indices of least important channels
        DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=idxs).prune()   # ⑩ prune the whole group
    return m                                                               # ⑪

# Shape check — only paths set to 1 should shrink
for d in [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 1)]:
    p = prune_paths(rn20, d)
    print(d, "| A", p.stem[0].out_channels, "· B", p.layer2[0].conv2.out_channels,
          "· C", p.layer3[0].conv2.out_channels, "· fc in", p.fc.in_features)

In [ ]:
# ▶ Cell 4 — Exp 1 (cumulative) · Exp 2 (single)
def run(paths):
    m = rn20 if paths == (0, 0, 0) else prune_paths(rn20, paths)
    macs, params = tp.utils.count_ops_and_params(m, EX_IN)
    return {"paths": str(paths), "macs_m": macs / 1e6, "params": int(params), "acc": acc_torch(m)}

# Exp 1: cumulative — how much more accuracy drops each time one more path is pruned
cum_df = pd.DataFrame([run(d) for d in [(0, 0, 0), (1, 0, 0), (1, 1, 0), (1, 1, 1)]])
cum_df["drop_from_prev"] = cum_df.acc.shift(1) - cum_df.acc

# Exp 2: single — how much accuracy drops vs. the original when only one path is pruned
solo_df = pd.DataFrame([run(d) for d in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]])
solo_df["drop_from_base"] = FP32_ACC - solo_df.acc          # always compared with the original

print("Exp 1 — cumulative"); display(cum_df.round(4))
print("Exp 2 — single"); display(solo_df.round(4))

In [ ]:
# ▶ Cell 5 (optional) — Single-path pruning: accuracy drop vs. MACs saved
base_macs = cum_df.macs_m[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(solo_df.paths, solo_df.drop_from_base * 100, color="tab:red"); ax[0].set_title("accuracy drop (%p)")
ax[1].bar(solo_df.paths, (1 - solo_df.macs_m / base_macs) * 100, color="tab:blue"); ax[1].set_title("MACs saved (%)")
plt.tight_layout(); plt.show()

In [ ]:
# ▶ Cell 6 (optional) — Ratio sweep per path: how far can each go?
rows = []
for k, name in enumerate(["A", "B", "C"]):
    for r in [0.1, 0.2, 0.3, 0.4, 0.5]:
        paths = tuple(1 if j == k else 0 for j in range(3))
        rows.append({"path": name, "ratio": r, "acc": acc_torch(prune_paths(rn20, paths, ratio=r))})
sweep_df = pd.DataFrame(rows).pivot(index="ratio", columns="path", values="acc")
display(sweep_df.round(4))

## Reading Part 1 results

| Table | Column | A large value means |
|---|---|---|
| Exp 1 (cumulative) | `drop_from_prev` | pruning this path **in addition** caused a large drop |
| Exp 2 (single) | `drop_from_base` | pruning this path **alone** causes a large drop = sensitive path |
| Both | `macs_m`, `params` | remaining compute and parameters |

Note that this sensitivity is measured **before recovery training**. Part 2 checks whether the same order holds after recovery.

---
# Part 2 — Stage-wise ratio pruning + FT / KD recovery

Part 1 pruned only the shortcut paths. Part 2 assigns **one ratio per stage (A, B, C)** and prunes
both that stage's shortcut path and the block-internal channels (conv1 outputs) **at the same ratio**.
This way the pruned scope matches the uniform baseline, which prunes every layer.

`ratios = (rA, rB, rC)` — e.g. `(0.2, 0.3, 0.4)` = 20% in stage A, 30% in B, 40% in C.

**Baseline**: uniform 20% pruning of all layers with torch-pruning `MetaPruner` (L1 magnitude, fc excluded).

**Comparison protocol**

- 5 levels (0.1–0.5) × 3 stages = 125 combinations; only those with **both** fewer MACs **and** fewer params than the baseline are candidates (83)
- 11 representative combinations: within each group of similar MACs (tier), only the pruning location (shape) changes,
  to answer "at the same compute, where is it better to prune?"
- Same training data, epochs, learning rate, eval set and seed for every combination
- Two references (MetaPruner 20%, stage-uniform 20%) trained with the same procedure

| Cell | Content | Time (T4, estimate) |
|---|---|---|
| 7 | Training data · `finetune` · `train_kd` · baseline function | tens of seconds |
| 8 | `prune_stages` + filtering 125 combinations | ~1 min |
| 9 | Representative combinations (editable) | instant |
| 10 | Per combination: prune → FT → KD (saved to CSV, resumable) | tens of seconds each |
| 11 | Results of this run | instant |
| 12 | Mean over runs → `results/summary.csv`, plots → `figures/` | instant |

The GPU is used automatically if available; otherwise it runs on CPU (much slower — cell 10 prints the remaining time).

**Repeated runs**: run cell 10 several times with `RUN_ID` = 1, 2, … Each run uses a different seed,
so you can see how much the results vary with training randomness.

In [ ]:
# ▶ Cell 7 — Training data (10,000 images) + training functions (fine-tuning · KD)
import cv2

tr = np.load(fetch("cifar_train10k.npz"))
_m = cv2.imdecode(tr["images_png"], cv2.IMREAD_COLOR)[..., ::-1]          # one PNG mosaic → RGB
Xtr_u8 = np.ascontiguousarray(_m.reshape(100, 32, 100, 32, 3).transpose(0, 2, 1, 3, 4).reshape(-1, 32, 32, 3))
ytr = tr["labels"]
print("Train set", Xtr_u8.shape, ytr.shape)

def batches(bs=128, seed=0):
    # shuffle + horizontal flip + 4-px pad & random crop (standard CIFAR augmentation)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(Xtr_u8))
    for i in range(0, len(idx), bs):
        b = Xtr_u8[idx[i:i + bs]]
        b = np.where(rng.random(len(b))[:, None, None, None] < 0.5, b[:, :, ::-1], b)
        pad = np.pad(b, ((0, 0), (4, 4), (4, 4), (0, 0)), mode="reflect"); dx, dy = rng.integers(0, 9, 2)
        yield torch.from_numpy(preprocess(pad[:, dy:dy + 32, dx:dx + 32])).to(DEV), torch.from_numpy(ytr[idx[i:i + bs]]).to(DEV)

STEPS_PER_EPOCH = len(Xtr_u8) // 128 + 1          # 10000/128 → 78 full batches + 1 partial = 79

def finetune(model, epochs=3, lr=1e-2, seed=0):
    model.to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * STEPS_PER_EPOCH)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in batches(seed=seed * 1000 + ep):    # different shuffling per run
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    return time.time() - t0

def soft(logits, T=1.0):
    return F.softmax(logits / T, dim=1)

def kd_loss(s_logits, t_logits, y, T=4.0, alpha=0.5):
    ce = F.cross_entropy(s_logits, y)                                     # match the ground truth
    kl = F.kl_div(F.log_softmax(s_logits / T, dim=1), soft(t_logits, T), reduction="batchmean")  # match the teacher distribution
    return alpha * ce + (1 - alpha) * T**2 * kl

def train_kd(student, teacher, epochs=3, lr=1e-2, T=4.0, alpha=0.5, seed=0):
    student.to(DEV); teacher = teacher.to(DEV).eval()                      # the teacher is not trained → keep in eval
    opt = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * STEPS_PER_EPOCH)
    t0 = time.time()
    for ep in range(epochs):
        student.train()
        for xb, yb in batches(seed=seed * 1000 + ep):    # different shuffling per run
            with torch.no_grad():
                t_out = teacher(xb)
            loss = kd_loss(student(xb), t_out, yb, T, alpha)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    student.eval()
    return time.time() - t0

def prune_channels(model, ratio):
    # Baseline: MetaPruner, prune every layer uniformly by `ratio` (fc excluded)
    m = copy.deepcopy(model).cpu().eval()
    tp.pruner.MetaPruner(m, EX_IN, importance=tp.importance.MagnitudeImportance(p=1),
                         pruning_ratio=ratio, ignored_layers=[m.fc]).step()
    return m

In [ ]:
# ▶ Cell 8 — One ratio per stage: prune the shortcut path and block internals together
STAGES = [  # (representative conv of the shortcut path, blocks of the stage)
    (lambda m: m.stem[0],          lambda m: list(m.layer1)),   # A
    (lambda m: m.layer2[0].conv2,  lambda m: list(m.layer2)),   # B
    (lambda m: m.layer3[0].conv2,  lambda m: list(m.layer3)),   # C
]

def prune_stages(model, ratios):
    m = copy.deepcopy(model).cpu().eval()
    DG = tp.DependencyGraph().build_dependency(m, example_inputs=EX_IN)
    imp = tp.importance.MagnitudeImportance(p=1)

    def cut(root, r):
        n_cut = int(round(root.out_channels * r))
        if n_cut == 0: return
        g = DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=list(range(root.out_channels)))
        idxs = imp(g).argsort()[:n_cut].tolist()                            # lowest-scoring channels first
        DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=idxs).prune()

    for r, (get_path, get_blocks) in zip(ratios, STAGES):
        if r <= 0: continue
        cut(get_path(m), r)                         # 1) shortcut path
        for blk in get_blocks(m): cut(blk.conv1, r) # 2) block internals (conv1 out = conv2 in)
    return m

def count(m):
    macs, params = tp.utils.count_ops_and_params(m, EX_IN)
    return macs / 1e6, int(params)

BASE_MACS, BASE_PARAMS = count(prune_channels(rn20, 0.2))
print(f"Baseline (uniform 20%): MACs {BASE_MACS:.2f}M | params {BASE_PARAMS:,}")

# Count size only for all 125 combinations (no accuracy, so it is fast)
import itertools
LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5]
grid = []
for rs in itertools.product(LEVELS, repeat=3):
    macs, params = count(prune_stages(rn20, rs))
    grid.append({"ratios": rs, "macs_m": macs, "params": params,
                 "pass": macs <= BASE_MACS and params <= BASE_PARAMS})
grid_df = pd.DataFrame(grid)
print(f"Combinations with both fewer MACs and params than the baseline: {grid_df['pass'].sum()} / {len(grid_df)}")

In [ ]:
# ▶ Cell 9 — Representative combinations (tier = similar MACs, shape = where to prune more)
#   uniform: same everywhere / increasing: more toward C / decreasing: more at A / middle: more at B
REPS = [
    ("~20M", "uniform",    (0.3, 0.3, 0.3)),
    ("~20M", "increasing", (0.2, 0.3, 0.4)),
    ("~20M", "decreasing", (0.4, 0.3, 0.2)),
    ("~20M", "middle",     (0.2, 0.5, 0.2)),
    ("~15M", "uniform",    (0.4, 0.4, 0.4)),
    ("~15M", "increasing", (0.3, 0.4, 0.5)),
    ("~15M", "decreasing", (0.5, 0.4, 0.3)),
    ("~15M", "middle",     (0.3, 0.5, 0.3)),
    ("~11M", "uniform",    (0.5, 0.5, 0.5)),
    ("~11M", "increasing", (0.4, 0.5, 0.5)),
    ("~11M", "decreasing", (0.5, 0.5, 0.4)),
]
REFS = [  # references (run regardless of the filter)
    ("baseline", "MetaPruner 20%", None),
    ("baseline", "stage 20%",      (0.2, 0.2, 0.2)),
]

lookup = grid_df.set_index("ratios")
for tier, shape, rs in REPS:
    row = lookup.loc[[rs]].iloc[0]
    print(f"{tier:5s} {shape:10s} {rs}  MACs {row.macs_m:6.2f}M  params {row.params:7,}  {'pass' if row['pass'] else '✗ larger than baseline'}")

In [ ]:
# ▶ Cell 10 — Experiment: prune → accuracy (as-is / fine-tuning / KD)
#   Saved to CSV after each combination → re-running skips finished ones
EPOCHS = 3
RUN_ID = 1             # ← run number (repeat with 1, 2, …)
SEED = RUN_ID - 1      # run 1 → seed 0, run 2 → seed 1
OUT = os.path.join(RESULTS_DIR, f"run{RUN_ID}.csv")

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
done_keys = set(done["name"]) if len(done) else set()
teacher = copy.deepcopy(rn20)          # keep rn20 untouched; the teacher is a copy
jobs = REFS + REPS
t_start = time.time(); n_new = 0

for i, (tier, shape, rs) in enumerate(jobs):
    name = f"{tier}/{shape}"
    if name in done_keys:
        print(f"[{i+1}/{len(jobs)}] {name} — already done, skipped"); continue

    pruned = prune_channels(rn20, 0.2) if rs is None else prune_stages(rn20, rs)
    macs, params = count(pruned)
    acc0 = acc_torch(pruned)

    torch.manual_seed(SEED); ft = copy.deepcopy(pruned); t_ft = finetune(ft, EPOCHS, seed=SEED); acc_ft = acc_torch(ft)
    torch.manual_seed(SEED); kd = copy.deepcopy(pruned); t_kd = train_kd(kd, teacher, EPOCHS, seed=SEED); acc_kd = acc_torch(kd)

    row = {"run": RUN_ID, "seed": SEED, "name": name, "tier": tier, "shape": shape, "ratios": str(rs), "macs_m": round(macs, 2), "params": params,
           "acc0": acc0, "acc_ft": acc_ft, "acc_kd": acc_kd, "t_ft": round(t_ft, 1), "t_kd": round(t_kd, 1)}
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(OUT, index=False)

    n_new += 1; left = sum(1 for t, s, _ in jobs[i+1:] if f"{t}/{s}" not in done_keys)
    eta = (time.time() - t_start) / n_new * left
    print(f"[{i+1}/{len(jobs)}] {name:28s} MACs {macs:5.2f}M | as-is {acc0:.3f} → FT {acc_ft:.3f} / KD {acc_kd:.3f}"
          f" | {t_ft + t_kd:.0f}s | ETA {eta/60:.1f} min")

print("Saved:", OUT)

In [ ]:
# ▶ Cell 11 — Results of this run
res = pd.read_csv(OUT)
res["kd_gain"] = res.acc_kd - res.acc_ft                       # how much more KD recovered than FT
res["gap_to_fp32"] = FP32_ACC - res[["acc_ft", "acc_kd"]].max(axis=1)
display(res[["tier", "shape", "ratios", "macs_m", "params", "acc0", "acc_ft", "acc_kd", "kd_gain", "gap_to_fp32"]].round(4))

reps = res[res.tier != "baseline"]
print("\nKD accuracy — tier × shape"); display(reps.pivot(index="shape", columns="tier", values="acc_kd").round(4))

In [ ]:
# ▶ Cell 12 — Average over runs → results/summary.csv, plots → figures/
#   run*.csv are raw records and are never modified; the mean goes to a separate file
import glob
runs = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(os.path.join(RESULTS_DIR, "run*.csv")))], ignore_index=True)
print("Runs merged:", sorted(runs.run.unique()))

keys = ["name", "tier", "shape", "ratios", "macs_m", "params"]          # identical across runs (pruning is deterministic)
summary = (runs.groupby(keys, dropna=False, sort=False)
               .agg(n_runs=("run", "nunique"),
                    acc0=("acc0", "mean"),
                    acc_ft=("acc_ft", "mean"), acc_ft_std=("acc_ft", "std"),
                    acc_kd=("acc_kd", "mean"), acc_kd_std=("acc_kd", "std"))
               .reset_index())
summary["kd_gain"] = summary.acc_kd - summary.acc_ft
summary["gap_to_fp32"] = FP32_ACC - summary.acc_kd
summary.round(4).to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
display(summary.round(4))

# plot labels (prefix references with "baseline:")
summary["label"] = np.where(summary.tier == "baseline", "baseline: " + summary["shape"], summary["shape"])
COLOR = {"uniform": "tab:gray", "increasing": "tab:blue", "decreasing": "tab:red", "middle": "tab:green"}

def pareto(df, x):
    # scan by increasing x, keep points more accurate than all before = no better alternative (Pareto front)
    best, keep = -1, []
    for i, r in df.sort_values(x).iterrows():
        if r.acc_kd > best: keep.append(i); best = r.acc_kd
    return df.loc[keep]

def plot(x, xlabel, fname):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    reps = summary[summary.tier != "baseline"]
    for lab, g in reps.groupby("label"):
        ax.scatter(g[x], g.acc_kd, color=COLOR[lab], label=lab, s=50)
        ax.scatter(g[x], g.acc_ft, color=COLOR[lab], marker="x", alpha=0.4)     # x marker = FT of the same combination
    for _, r in summary[summary.tier == "baseline"].iterrows():
        ax.scatter(r[x], r.acc_kd, marker="*", s=220, color="k")
        ax.annotate(r.label, (r[x], r.acc_kd), textcoords="offset points", xytext=(6, -12), fontsize=8)
    p = pareto(reps, x); ax.plot(p[x], p.acc_kd, "k--", lw=1, alpha=0.6, label="Pareto front (KD)")
    ax.axhline(FP32_ACC, ls=":", color="gray", label=f"unpruned {FP32_ACC:.3f}")
    ax.set_xlabel(xlabel); ax.set_ylabel("accuracy (KD: dot, FT: x)")
    ax.set_title(f"{xlabel} vs accuracy  (mean of {summary.n_runs.max()} runs)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
    path = os.path.join(FIG_DIR, fname); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
    print("Saved:", path)

plot("macs_m", "MACs (M)", "pareto_macs.png")
plot("params", "params", "pareto_params.png")

## Reading Part 2 results

| Column | Meaning |
|---|---|
| `acc0` | accuracy right after pruning, before recovery training |
| `acc_ft` / `acc_kd` | accuracy recovered by fine-tuning / KD |
| `kd_gain` | KD − FT. Positive means learning from the teacher's (original model's) output distribution helped |
| `gap_to_fp32` | remaining gap to the original model's accuracy |
| `*_std` | standard deviation across runs (variation from training randomness) |

- Compare shapes **within a tier** → at the same compute, where is it better to prune?
- Points left of the **★ baseline** (less compute) at similar or higher accuracy → more efficient than uniform pruning
- How recovery degrades as the tier goes down (~20M → ~11M) → how far can we compress?

**Caveat**: the eval set has 2,000 images, so one image = 0.05%p and the standard error near 0.8 accuracy is about ±0.9%p.
Repeating runs reduces only training randomness, not the error from the eval-set size.

### Ideas for extension

- Increase `EPOCHS` and check whether KD's advantage persists
- Add combinations in cell 9 — pick from `grid_df[grid_df["pass"]]`
- Change KD hyperparameters → `train_kd(kd, teacher, EPOCHS, T=2.0, alpha=0.3)`